# Digital Image Processing - Assignment 0

In [1]:
import cv2
import numpy as np

## 1. Read an image file into an array

In [2]:
def read_image(filename):
    img = cv2.imread(filename, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Unable to read image: {filename}")
    return img

img_array = read_image("input\\input_colored.jpg")
img_shape = img_array.shape
img_type = img_array.dtype

print(img_shape)
print(img_type)
print(img_array)

(2353, 4000, 3)
uint8
[[[ 59 136 109]
  [100 138 116]
  [113 142 119]
  ...
  [ 71  65  88]
  [ 68  61  88]
  [ 66  61  92]]

 [[ 61 137 113]
  [ 99 138 117]
  [110 141 120]
  ...
  [ 68  70  81]
  [ 63  65  76]
  [ 58  64  77]]

 [[ 61 136 114]
  [ 95 134 113]
  [101 137 115]
  ...
  [ 67  78  76]
  [ 58  68  75]
  [ 54  64  82]]

 ...

 [[102 126 138]
  [ 98 129 126]
  [ 98 133 119]
  ...
  [ 37  41  30]
  [ 41  44  35]
  [ 45  47  41]]

 [[102 131 128]
  [101 131 132]
  [ 99 128 132]
  ...
  [ 37  43  32]
  [ 42  46  35]
  [ 48  49  39]]

 [[ 98 120 145]
  [102 121 134]
  [ 91 123 136]
  ...
  [ 35  42  29]
  [ 40  44  25]
  [ 44  40  29]]]


## 2. Write an array into an image file
Note: Make sure you can handle colour and grayscale images

In [3]:
def write_image(filename, image_array):
    cv2.imwrite(filename, image_array)


rand_array = np.random.randint(-255, 255, size=img_shape).astype(np.int16)
image_array = np.clip(rand_array + img_array.astype(np.int16), 0, 255).astype(img_type)
write_image("output\\output_random.jpg", image_array)


## 3. Change brightness of the image.

In [4]:
def change_brightness(image_array, value):
    new_image = image_array.astype(np.int16) + value
    new_image = np.clip(new_image, 0, 255)
    new_image = new_image.astype(img_type)
    return new_image

brighter = change_brightness(img_array, 50)
write_image("brightness\\output_brighter.jpg", brighter)

darker = change_brightness(img_array, -50)
write_image("brightness\\output_darker.jpg", darker)

## 4. Change contrast of the image

In [5]:
def change_contrast(image_array, factor):
    new_image = factor * (image_array)
    new_image = np.clip(new_image, 0, 255)
    return new_image.astype(img_type)

more_contrast = change_contrast(img_array, 1.25)
write_image("contrast\\output_more_contrast.jpg", more_contrast)

less_contrast = change_contrast(img_array, 0.75)
write_image("contrast\\output_less_contrast.jpg", less_contrast)

## 5. Change a colour image to grayscale.
Qn: Are there different ways of doing this? What is the visual effect of each?

In [6]:
def grayscale_average(img):
    return np.mean(img, axis=2).astype(img_type)

def grayscale_luminosity(img):
    return (0.3*img[:,:,0] + 0.59*img[:,:,1] + 0.11*img[:,:,2]).astype(img_type)

def grayscale_lightness(img):
    max_val = np.max(img, axis=2)
    min_val = np.min(img, axis=2)
    return ((max_val + min_val) / 2).astype(img_type)

def grayscale_single_channel(img, channel=0):
    return img[:,:,channel]

###############################################################################

average_gray = grayscale_average(img_array)
write_image("grayscale\\output_gray_average.jpg", average_gray)

luminosity_gray = grayscale_luminosity(img_array)
write_image("grayscale\\output_gray_luminosity.jpg", luminosity_gray)

lightness_gray = grayscale_lightness(img_array)
write_image("grayscale\\output_gray_lightness.jpg", lightness_gray)

###############################################################################

red_channel = grayscale_single_channel(img_array, channel=0)
write_image("grayscale\\output_red_channel.jpg", red_channel)

green_channel = grayscale_single_channel(img_array, channel=1)
write_image("grayscale\\output_green_channel.jpg", green_channel)

blue_channel = grayscale_single_channel(img_array, channel=2)
write_image("grayscale\\output_blue_channel.jpg", blue_channel)

## 6. Convert a grayscale image to colour using a pseudo colour mapping.


In [7]:
def grayscale_to_pseudo(gray_img):
    gray_img = gray_img.astype(img_type)

    h, w = gray_img.shape
    color_img = np.zeros((h, w, 3), dtype=img_type)

    gray = gray_img.astype(np.float32) / 255.0

    color_img[:, :, 0] = np.clip(255 * np.maximum(1.5 - np.abs(4 * gray - 1), 0), 0, 255).astype(img_type)
    color_img[:, :, 1] = np.clip(255 * np.maximum(1.5 - np.abs(4 * gray - 2), 0), 0, 255).astype(img_type)
    color_img[:, :, 2] = np.clip(255 * np.maximum(1.5 - np.abs(4 * gray - 3), 0), 0, 255).astype(img_type)

    return color_img

pseudo_colored_average = grayscale_to_pseudo(average_gray)
write_image("pseudocolored\\output_pseudo_colored_average_image.jpg", pseudo_colored_average)

pseudo_colored_luminosity = grayscale_to_pseudo(luminosity_gray)
write_image("pseudocolored\\output_pseudo_colored_luminosity_image.jpg", pseudo_colored_luminosity)

pseudo_colored_image = grayscale_to_pseudo(lightness_gray)
write_image("pseudocolored\\output_pseudo_colored_lightness_image.jpg", pseudo_colored_image)

## 7. Read in an image with green screen for background and a second one for background. The function should replace the green screen with the corresponding pixels from the background image

In [8]:
def detect_green_pixels(image, green_thresh):
    R = image[:, :, 0].astype(np.int16)
    G = image[:, :, 1].astype(np.int16)
    B = image[:, :, 2].astype(np.int16)
    mask = (G - np.maximum(R, B)) >= green_thresh
    return mask


def resize_to_match(img, target_shape):
    target_h, target_w = target_shape[:2]
    h, w = img.shape[:2]
    
    # Calculate scaling factors
    scale_h = h / target_h
    scale_w = w / target_w
    
    # Create output array
    if len(img.shape) == 3:
        resized = np.zeros((target_h, target_w, img.shape[2]), dtype=img.dtype)
    else:
        resized = np.zeros((target_h, target_w), dtype=img.dtype)
    
    # Bilinear interpolation
    for y in range(target_h):
        for x in range(target_w):
            src_y = y * scale_h
            src_x = x * scale_w
            
            y0, y1 = int(src_y), min(int(src_y) + 1, h - 1)
            x0, x1 = int(src_x), min(int(src_x) + 1, w - 1)
            
            fy = src_y - y0
            fx = src_x - x0
            
            resized[y, x] = (
                (1 - fy) * (1 - fx) * img[y0, x0] +
                (1 - fy) * fx * img[y0, x1] +
                fy * (1 - fx) * img[y1, x0] +
                fy * fx * img[y1, x1]
            ).astype(img.dtype)
    
    return resized

def replace_green_screen(foreground, background, green_thresh=200):
    mask = detect_green_pixels(foreground, green_thresh)
    
    background_resized = resize_to_match(background, foreground.shape)
    
    result = foreground.copy()
    result[mask] = background_resized[mask]
    return result


foreground_image = read_image('input\\input_bg.jpg')
background_image = read_image('input\\input_colored.jpg')

result_image = replace_green_screen(foreground_image, background_image)
write_image('greenscreen\\merged_image.jpg', result_image)

## 8. Read a video file and convert into an sequence (array) of images, and also to write it back as a video.

In [9]:
def video_to_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    cap.release()
    return frames

def frames_to_video(frames, output_path, fps=30):
    if not frames:
        raise ValueError("No frames to write!")

    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for frame in frames:
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        out.write(frame_bgr)

    out.release()

frames = video_to_frames("input\\video.mp4")

print(f"Extracted {len(frames)} frames")
frames_to_video(frames, "output\\video.mp4", fps=10)


Extracted 64 frames


## 9. Create a 1 second transition video (fade, slide, any other) from one image to another.

In [10]:
def fade_transition(img1, img2, frames_count):
    frames = []

    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)

    for i in range(frames_count):
        alpha = i / (frames_count - 1)
        frame = (1 - alpha) * img1 + alpha * img2
        frames.append(frame.astype(np.uint8))

    return frames


def checkerboard_transition(img1, img2, frames_count, squares=8):
    frames = []

    height, width, _ = img1.shape
    block_w = width // squares
    block_h = height // squares

    for i in range(frames_count):
        progress = i / (frames_count - 1)

        frame = img1.copy()

        for y in range(squares):
            for x in range(squares):

                appear_time = (x + y) / (squares * 2)

                if progress >= appear_time:

                    x_start = x * block_w
                    y_start = y * block_h

                    frame[
                        y_start:y_start + block_h,
                        x_start:x_start + block_w
                    ] = img2[
                        y_start:y_start + block_h,
                        x_start:x_start + block_w
                    ]

        frames.append(frame)

    return frames


def slide_transition(img1, img2, frames_count):
    frames = []

    height, width, _ = img1.shape

    for i in range(frames_count):
        progress = i / (frames_count - 1)
        offset = int(progress * width)

        frame = np.zeros_like(img1)

        if offset < width:
            frame[:, :width - offset] = img1[:, offset:]
            frame[:, width - offset:] = img2[:, :offset]

        frames.append(frame)

    return frames


def create_transition_video(img1, img2, transition, output_path, fps=30):
    frames_count = fps

    if transition == "fade":
        frames = fade_transition(img1, img2, frames_count)

    elif transition == "checkerboard":
        frames = checkerboard_transition(img1, img2, frames_count)

    elif transition == "slide":
        frames = slide_transition(img1, img2, frames_count)

    else:
        raise ValueError("Unknown transition")

    # read_image gives BGR, frames_to_video expects RGB
    frames = [
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        for frame in frames
    ]

    frames_to_video(frames, output_path, fps)


image1 = read_image("input\\Image_1.jpg")
image2 = read_image("input\\Image_2.jpg")

image2 = resize_to_match(image2, image1.shape)

create_transition_video(
    image1,
    image2,
    "fade",
    "output\\fade.mp4"
)

create_transition_video(
    image1,
    image2,
    "checkerboard",
    "output\\checkerboard.mp4"
)

create_transition_video(
    image1,
    image2,
    "slide",
    "output\\slide.mp4"
)